In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, HBox, Layout, VBox, HTML
from IPython.display import display

def plot_bs_filter_specs(epsilon=0.4, A=10.0, wp1=1.8, ws1=2.5, ws2=4.5, wp2=5.2):
    fig, ax = plt.subplots(figsize=(10, 6))
    omega = np.linspace(0.01, 7.0, 1000)
    n = 4

    def T_n(n, x): 
        return np.where(x <= 1.0, np.cos(n * np.arccos(np.clip(x, -1.0, 1.0))), 
                                  np.cosh(n * np.arccosh(np.maximum(x, 1.0))))

    H_mag = np.zeros_like(omega)

    for i, w in enumerate(omega):
        if w <= wp1 or w >= wp2:
            center = wp1 if w <= wp1 else wp2
            w_mapped = np.abs(w - center) / 0.5
            val = 1.0 / np.sqrt(1.0 + (epsilon * T_n(n, w_mapped))**2)
            H_mag[i] = min(max(val, 1.0 / np.sqrt(1.0 + epsilon**2)), 1.0)
        elif ws1 <= w <= ws2:
            val = 1.0 / A * (1.0 / (1.0 + 0.5 * np.abs(T_n(n, np.abs(w - (ws1 + ws2) / 2) / ((ws2 - ws1) / 2)))))
            H_mag[i] = min(val, 1.0 / A)
        else:
            if w < ws1:
                ratio = (w - wp1) / (ws1 - wp1)
                val_wp = 1.0 / np.sqrt(1.0 + epsilon**2)
                val_ws = 1.0 / A
                H_mag[i] = val_wp - ratio * (val_wp - val_ws)
            else:
                ratio = (w - ws2) / (wp2 - ws2)
                val_ws = 1.0 / A
                val_wp = 1.0 / np.sqrt(1.0 + epsilon**2)
                H_mag[i] = val_ws + ratio * (val_wp - val_ws)

    ax.plot(omega, H_mag, 'r-', linewidth=2, label=r'$|H(e^{j\omega})|$')

    upper_bound_pb = 1.0
    lower_bound_pb = 1.0 / np.sqrt(1.0 + epsilon**2)
    stop_bound = 1.0 / A

    ax.hlines(upper_bound_pb, 0, wp1, colors='k', linewidth=1.5)
    ax.hlines(lower_bound_pb, 0, wp1, colors='k', linewidth=1.5, linestyle='--')
    ax.hlines(stop_bound, ws1, ws2, colors='k', linewidth=1.5)
    ax.hlines(upper_bound_pb, wp2, omega[-1], colors='k', linewidth=1.5)
    ax.hlines(lower_bound_pb, wp2, omega[-1], colors='k', linewidth=1.5, linestyle='--')

    ax.axvline(wp1, ymin=0.3, ymax=1.0, color='k', linestyle='--', linewidth=1)
    ax.axvline(ws1, ymin=0, ymax=0.3, color='k', linestyle='--', linewidth=1)
    ax.axvline(ws2, ymin=0, ymax=0.3, color='k', linestyle='--', linewidth=1)
    ax.axvline(wp2, ymin=0.3, ymax=1.0, color='k', linestyle='--', linewidth=1)

    # ΔΙΟΡΘΩΣΗ: Σκίαση μέσα στις ζώνες διέλευσης (passbands) ανάμεσα σε lower_bound_pb και 1.0
    ax.fill_between([0, wp1], lower_bound_pb, upper_bound_pb, color='blue', alpha=0.1, hatch='//')
    # Σκίαση μέσα στη ζώνη αποκοπής (stopband) ανάμεσα στο 0 και το stop_bound
    ax.fill_between([ws1, ws2], 0, stop_bound, color='blue', alpha=0.1, hatch='//')
    # Σκίαση μέσα στη δεύτερη ζώνη διέλευσης (passband 2) ανάμεσα σε lower_bound_pb και 1.0
    ax.fill_between([wp2, omega[-1]], lower_bound_pb, upper_bound_pb, color='blue', alpha=0.1, hatch='//')

    ax.set_xlim(0, max(omega))
    ax.set_ylim(-0.02, 1.25)
    ax.set_xlabel(r'$\omega$', fontsize=14)
    ax.set_ylabel(r'$|H(e^{j\omega})|$', fontsize=14)
    ax.set_xticks([wp1, ws1, ws2, wp2])
    ax.set_xticklabels([r'$\omega_{p1}$', r'$\omega_{s1}$', r'$\omega_{s2}$', r'$\omega_{p2}$'], fontsize=12)

    y_ticks = [stop_bound, lower_bound_pb, 1.0]
    y_labels = [r'$\frac{1}{A}$', r'$\frac{1}{\sqrt{1+\varepsilon^2}}$', '1']
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=12)

    ax.text(wp1 / 2, 0.5, 'Passband 1', color='green', fontsize=9, fontweight='bold', ha='center')
    ax.text((ws1 + ws2) / 2, 0.5, 'Stopband', color='green', fontsize=10, fontweight='bold', ha='center')
    ax.text(wp2 + (omega[-1] - wp2) / 2, 0.5, 'Passband 2', color='green', fontsize=9, fontweight='bold', ha='center')

    ax.grid(True, linestyle=':', alpha=0.6)
    plt.title('Normalized Frequency Response of a Band-Stop Analog Filter', fontsize=13, pad=15)
    plt.show()

slider_layout = Layout(width='260px')
style_opts = {'description_width': '55px'}

eps_slider = FloatSlider(min=0.1, max=1.5, step=0.05, value=0.4, description='ε:', style=style_opts, layout=slider_layout)
A_slider = FloatSlider(min=5.0, max=30.0, step=1.0, value=10.0, description='A:', style=style_opts, layout=slider_layout)
wp1_slider = FloatSlider(min=0.5, max=2.0, step=0.1, value=1.8, description='ωp1:', style=style_opts, layout=slider_layout)
ws1_slider = FloatSlider(min=2.1, max=3.0, step=0.1, value=2.5, description='ωs1:', style=style_opts, layout=slider_layout)
ws2_slider = FloatSlider(min=3.8, max=4.8, step=0.1, value=4.5, description='ωs2:', style=style_opts, layout=slider_layout)
wp2_slider = FloatSlider(min=4.9, max=6.5, step=0.1, value=5.2, description='ωp2:', style=style_opts, layout=slider_layout)

widget_plot = interactive(plot_bs_filter_specs, epsilon=eps_slider, A=A_slider, wp1=wp1_slider, ws1=ws1_slider, ws2=ws2_slider, wp2=wp2_slider)

theory_html = HTML("""
<div style="font-family: monospace; font-size: 13px; line-height: 1.5; margin-bottom: 8px;">
<b>ε:</b> Passband ripple parameter.<br>
<b>A:</b> Stopband attenuation factor.<br>
<b>ωp1, ωp2:</b> Passband edge frequencies.<br>
<b>ωs1, ωs2:</b> Stopband edge frequencies.
</div>
""")

controls = VBox([eps_slider, A_slider, wp1_slider, ws1_slider, ws2_slider, wp2_slider], layout=Layout(width='280px', justify_content='center'))

main_layout = HBox([widget_plot.children[-1], controls], layout=Layout(width='1100px', align_items='center', justify_content='center'))

display(VBox([theory_html, main_layout], layout=Layout(width='1100px')))